# Plantilla de Ejercicio — Módulo 3

---
## Antes de empezar — responde estas preguntas

Escribe tus respuestas aquí (doble clic para editar):

**1. ¿Qué quiero predecir?**
> _Escribe aquí. Ejemplo: si un correo es spam o no_

**2. ¿La salida (Y) es...?**
> - [ ] Un número continuo (precio, temperatura) → usar **Regresión Lineal**
> - [ ] Una categoría (sí/no, clase A/B/C) → usar **Reg. Logística, Árbol o SVM**
> - [ ] No sé las categorías de antemano → usar **K-Means**

**3. ¿Cuál es el dataset?**
> _Escribe aquí. Ejemplo: Breast Cancer de sklearn, id=17 de UCI, archivo datos.csv_

**4. ¿Cuántas características tiene X?**
> _Lo verás en X.shape cuando cargues los datos_

**5. ¿Cuál es el modelo que voy a usar?**
> _Escribe aquí_

---

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 1 — Imports
# Corre siempre primero. Importa todo lo que puedas necesitar.
# ═══════════════════════════════════════════════════════════════
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# División y escalado — siempre necesario
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler

# Modelos — descomenta el que vas a usar
from sklearn.linear_model import LinearRegression    # para predecir números
from sklearn.linear_model import LogisticRegression  # para clasificar
from sklearn.tree         import DecisionTreeClassifier  # árbol interpretable
from sklearn.svm          import SVC                  # máquina de soporte vectorial
from sklearn.cluster      import KMeans               # agrupación sin etiquetas

# Métricas
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,  # clasificación
    mean_squared_error, r2_score,                             # regresión
    silhouette_score                                           # clustering
)

# Ruta relativa al machote
RUTA = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'Machote'))
if RUTA not in sys.path:
    sys.path.append(RUTA)

print("Imports listos ✅")

---
## Paso 1 — Cargar los datos
Elige UNA opción según de dónde vienen tus datos.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 2 — Cargar datos
# Descomenta el bloque que aplica a tu ejercicio
# ═══════════════════════════════════════════════════════════════

# ── OPCIÓN A: sklearn (para practicar) ────────────────────────
from sklearn.datasets import load_breast_cancer  # ← cambia por load_iris, load_wine, load_diabetes, etc.
dataset     = load_breast_cancer()
X           = pd.DataFrame(dataset.data, columns=dataset.feature_names)
Y           = dataset.target
CLASES      = list(dataset.target_names)  # quita esto si es regresión

# ── OPCIÓN B: UCI (requiere internet) ─────────────────────────
# from ucimlrepo import fetch_ucirepo
# dataset  = fetch_ucirepo(id=17)                # ← cambia el id
# X        = dataset.data.features
# Y_raw    = dataset.data.targets
# print("Columnas en Y:", Y_raw.columns.tolist()) # ver nombre exacto
# Y        = Y_raw['Diagnosis'].map({'M': 0, 'B': 1}).to_numpy()
# CLASES   = ['Maligno', 'Benigno']

# ── OPCIÓN C: CSV propio ──────────────────────────────────────
# df       = pd.read_csv(r"ruta/a/tu/archivo.csv")
# print(df.head())             # ver primeras filas
# print(df.columns.tolist())   # ver nombres de columnas
# X        = df.drop(columns=['nombre_columna_Y'])
# Y        = df['nombre_columna_Y'].values
# CLASES   = ['Clase0', 'Clase1']  # ajusta

print(f"Dataset cargado ✅")
print(f"  Muestras (filas):      {X.shape[0]}")
print(f"  Características (cols): {X.shape[1]}")
if 'CLASES' in dir():
    print(f"  Clases: {CLASES}")

---
## Paso 2 — Explorar los datos (EDA)
**SIEMPRE antes de modelar.** Muchos errores se evitan mirando los datos primero.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3 — Resumen rápido
# ═══════════════════════════════════════════════════════════════
print("="*55)
print("  ¿HAY VALORES FALTANTES (NaN)?")
print("="*55)
nan_total = X.isnull().sum().sum()
if nan_total == 0:
    print("  Ninguno ✅  (puedes continuar)")
else:
    print(f"  Hay {nan_total} NaN ⚠️  (hay que manejarlos antes de entrenar)")
    print("  Columnas afectadas:")
    print(X.isnull().sum()[X.isnull().sum() > 0])
    print()
    print("  Opciones para manejar NaN:")
    print("  X = X.dropna()            # eliminar filas con NaN")
    print("  X = X.fillna(X.mean())    # rellenar con la media")

print()
print("="*55)
print("  DISTRIBUCIÓN DE Y")
print("="*55)
try:
    vals, cnts = np.unique(Y, return_counts=True)
    for v, c in zip(vals, cnts):
        barra  = '█' * int(c / len(Y) * 25)
        nombre = CLASES[int(v)] if 'CLASES' in dir() and int(v) < len(CLASES) else str(v)
        alerta = " ⚠️ DESBALANCE" if c / len(Y) > 0.80 else ""
        print(f"  {nombre}: {c} ({c/len(Y)*100:.1f}%) {barra}{alerta}")
except:
    print(f"  Y continuo — min={Y.min():.2f}, max={Y.max():.2f}, media={Y.mean():.2f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 4 — Ver los datos
# ═══════════════════════════════════════════════════════════════
print("Primeras 5 filas:")
display(X.head())
print("\nEstadísticas básicas:")
display(X.describe())

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 5 — Correlación con Y
# Muestra qué columnas son útiles para predecir
# ═══════════════════════════════════════════════════════════════
df_temp = X.copy()
df_temp['__Y__'] = Y
correlaciones = df_temp.corr()['__Y__'].drop('__Y__').sort_values(key=abs, ascending=False)

UMBRAL = 0.5  # ← ajusta si necesitas más o menos columnas

fig, ax = plt.subplots(figsize=(8, max(4, len(correlaciones) * 0.3)))
colors = ['#D85A30' if abs(v) >= UMBRAL else '#aaa' for v in correlaciones]
correlaciones.plot(kind='barh', ax=ax, color=colors)
ax.axvline(x=0,       color='gray',  linewidth=0.8)
ax.axvline(x=UMBRAL,  color='green', linewidth=1.2, linestyle='--', label=f'umbral ±{UMBRAL}')
ax.axvline(x=-UMBRAL, color='green', linewidth=1.2, linestyle='--')
ax.set_title(f'Correlación con Y (naranja = útiles, gris = posible ruido)')
ax.legend()
plt.tight_layout()
plt.show()

# Dirección — importante para entender el dataset
prom = correlaciones.mean()
print(f"\nCorrelación promedio: {prom:+.4f}")
if prom < 0:
    print("→ Mayoría NEGATIVA: valores altos en X tienden a Y BAJO")
else:
    print("→ Mayoría POSITIVA: valores altos en X tienden a Y ALTO")

---
## Paso 3 — Preparar los datos

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 6 — Seleccionar características y dividir
# ═══════════════════════════════════════════════════════════════
features_utiles = correlaciones[correlaciones.abs() >= UMBRAL].index.tolist()
X_red = X[features_utiles]
print(f"Características seleccionadas: {len(features_utiles)} de {X.shape[1]}")

# Dividir
# Si es clasificación: stratify=Y  (mantiene proporción de clases)
# Si es regresión:     stratify=None
# Si es clustering:    no dividir (K-Means usa todos los datos)
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X_red, Y,
        test_size=0.25,
        stratify=Y,       # ← cambia a None si es regresión
        random_state=42
    )
    print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")
except Exception as e:
    print(f"Error con stratify: {e}")
    print("Intentando sin stratify (para regresión)...")
    X_train, X_test, y_train, y_test = train_test_split(
        X_red, Y, test_size=0.25, random_state=42
    )
    print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 7 — Escalar
# ═══════════════════════════════════════════════════════════════
# Siempre escalar DESPUÉS de dividir, y fit SOLO con train
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Datos escalados ✅")
print(f"  Media train (debe ser ~0): {X_train_sc.mean():.6f}")

---
## Paso 4 — Crear y entrenar el modelo
**Descomenta el bloque que corresponde al modelo que vas a usar.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 8 — Elegir modelo y entrenar
# Descomenta UNO de los siguientes bloques
# ═══════════════════════════════════════════════════════════════

# ── Regresión Lineal (para Y numérico continuo) ───────────────
# modelo = LinearRegression()
# modelo.fit(X_train_sc, y_train)
# TIPO = 'regresion'

# ── Regresión Logística (clasificación binaria/multiclase) ────
modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train_sc, y_train)
TIPO = 'clasificacion'

# ── Árbol de Decisión (clasificación interpretable) ───────────
# modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
# modelo.fit(X_train_sc, y_train)
# TIPO = 'clasificacion'

# ── SVM (clasificación robusta) ───────────────────────────────
# modelo = SVC(kernel='rbf', C=1.0, random_state=42)
# modelo.fit(X_train_sc, y_train)
# TIPO = 'clasificacion'

# ── K-Means (sin etiquetas) ───────────────────────────────────
# modelo = KMeans(n_clusters=2, random_state=42, n_init=10)
# modelo.fit(X_train_sc)   # ← sin y_train
# TIPO = 'clustering'

print(f"Modelo entrenado: {type(modelo).__name__} ✅")

---
## Paso 5 — Predecir y evaluar

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 9 — Evaluación según el tipo de modelo
# ═══════════════════════════════════════════════════════════════

if TIPO == 'clasificacion':
    Y_pred = modelo.predict(X_test_sc)
    acc = accuracy_score(y_test, Y_pred)

    print(f"Accuracy: {acc:.4f} ({acc*100:.1f}%)\n")
    print("Reporte completo:")
    nombres = CLASES if 'CLASES' in dir() else None
    print(classification_report(y_test, Y_pred, target_names=nombres))

    cm = confusion_matrix(y_test, Y_pred)
    etiquetas = [f'Pred {c}' for c in (CLASES if 'CLASES' in dir() else range(len(np.unique(Y))))]
    y_etiquetas = [f'Real {c}' for c in (CLASES if 'CLASES' in dir() else range(len(np.unique(Y))))]
    fig, ax = plt.subplots(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=etiquetas, yticklabels=y_etiquetas, ax=ax)
    ax.set_title(f'Matriz de Confusión — {acc*100:.1f}%')
    plt.tight_layout()
    plt.show()

elif TIPO == 'regresion':
    Y_pred = modelo.predict(X_test_sc)
    mse  = mean_squared_error(y_test, Y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test, Y_pred)

    print(f"MSE:  {mse:.4f}")
    print(f"RMSE: {rmse:.4f}  (el modelo se equivoca ±{rmse:.2f} unidades)")
    print(f"R²:   {r2:.4f}  (explica el {r2*100:.1f}% de la variación)")

    fig, ax = plt.subplots(figsize=(5,5))
    ax.scatter(y_test, Y_pred, alpha=0.5, s=20, color='#3B8BD4')
    lims = [min(y_test.min(), Y_pred.min()), max(y_test.max(), Y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfecto')
    ax.set_xlabel('Real')
    ax.set_ylabel('Predicho')
    ax.set_title(f'Real vs Predicho — R²={r2:.3f}')
    ax.legend()
    plt.tight_layout()
    plt.show()

elif TIPO == 'clustering':
    clusters = modelo.predict(X_test_sc)
    sil = silhouette_score(X_test_sc, clusters)
    print(f"Silhouette score: {sil:.4f}")
    print(f"(-1 a +1 → más cerca de 1 = clusters mejor separados)")
    print(f"Muestras por cluster: {dict(zip(*np.unique(clusters, return_counts=True)))}")

---
## Paso 6 — Conclusiones
Escribe aquí tus conclusiones (doble clic para editar).

**¿Qué accuracy / R² / Silhouette obtuviste?**
> _Escribe aquí_

**¿El modelo funciona bien para este problema?**
> _Escribe aquí_

**¿Qué podrías hacer para mejorar el modelo?**
> - Probar con más/menos features (cambiar el umbral de correlación)
> - Probar otro modelo
> - Ajustar hiperparámetros (max_depth, C, K, etc.)
> - _Escribe tus ideas aquí_